In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import os, time, copy, json
from datetime import datetime
import matplotlib.pyplot as plt

# Move to project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# ==========================================================================
# MLOPS LOGGING & HYPERPARAMETERS
# ==========================================================================
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join("experiments", "runs", f"run_{TIMESTAMP}")
MODELS_DIR = os.path.join("experiments", "models")
PROCESSED_DIR = os.path.join("data", "processed")

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

PARAMS = {
    "PATCH_WIDTH": 64,
    "PATCH_STRIDE": 32,
    "BATCH_SIZE": 128,
    "LEARNING_RATE": 1e-3,
    "WEIGHT_DECAY": 1e-5,
    "NUM_EPOCHS": 150,
    "PATIENCE": 20,
    "BOTTLENECK_DIM": 64,
    "LOSS_FUNCTION": "L1Loss",
    "DENOISING_FACTOR": 0.1,
    "MIXUP_ENABLED": True
}

# Save parameters to run folder
with open(os.path.join(RUN_DIR, "run_params.json"), "w") as f:
    json.dump(PARAMS, f, indent=4)

# ==========================================================================
# ARCHITECTURE (True Bottleneck + Spatial Dropout)
# ==========================================================================
class CNNAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = self._conv_block(1, 32)
        self.enc2 = self._conv_block(32, 64)
        self.enc3 = self._conv_block(64, 128)
        self.enc4 = self._conv_block(128, 128)
        
        # True Bottleneck (4096 -> 64 -> 4096)
        self.flatten = nn.Flatten()
        self.bottleneck_encode = nn.Linear(128 * 8 * 4, PARAMS["BOTTLENECK_DIM"]) 
        self.bottleneck_decode = nn.Linear(PARAMS["BOTTLENECK_DIM"], 128 * 8 * 4) 

        # Decoder
        self.dec1 = self._deconv_block(128, 128)
        self.dec2 = self._deconv_block(128, 64)
        self.dec3 = self._deconv_block(64, 32)
        self.dec4 = nn.Sequential(
            nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid() 
        )

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.2) # Spatial Dropout for Domain Shift
        )

    def _deconv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        
        # Compress & Expand
        flat = self.flatten(e4)
        compressed = self.bottleneck_encode(flat)
        expanded = self.bottleneck_decode(compressed)
        d_input = expanded.view(-1, 128, 8, 4)
        
        d1 = self.dec1(d_input)
        d2 = self.dec2(d1)
        d3 = self.dec3(d2)
        return self.dec4(d3)

# Helper: Extract Patches
def extract_patches(data, width=PARAMS["PATCH_WIDTH"], stride=PARAMS["PATCH_STRIDE"]):
    patches = []
    for spec in data:
        for start in range(0, spec.shape[1] - width + 1, stride):
            patches.append(spec[:, start:start+width])
    return np.array(patches)

# ==========================================================================
# TRAINING LOOP
# ==========================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | Saving to: {RUN_DIR}")

machines = sorted([d for d in os.listdir(PROCESSED_DIR) if os.path.isdir(os.path.join(PROCESSED_DIR, d))])

for machine in machines:
    print(f"\n{'='*50}\nTRAINING: {machine}\n{'='*50}")
    
    train_path = os.path.join(PROCESSED_DIR, machine, "X_train.npy")
    val_path   = os.path.join(PROCESSED_DIR, machine, "X_val.npy")
    if not os.path.exists(train_path): continue
    
    X_train = extract_patches(np.load(train_path))
    X_val   = extract_patches(np.load(val_path))
    
    X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
    X_val_t   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)
    
    train_loader = DataLoader(TensorDataset(X_train_t, X_train_t), batch_size=PARAMS["BATCH_SIZE"], shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val_t, X_val_t), batch_size=PARAMS["BATCH_SIZE"], shuffle=False)
    
    model = CNNAutoencoder().to(device)
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=PARAMS["LEARNING_RATE"], weight_decay=PARAMS["WEIGHT_DECAY"])
    
    best_val_loss = float('inf')
    patience_counter = 0
    history_train, history_val = [], []
    
    for epoch in range(1, PARAMS["NUM_EPOCHS"] + 1):
        model.train()
        train_loss = 0
        for batch, _ in train_loader:
            batch = batch.to(device)
            
            # --- MIXUP AUGMENTATION ---
            if PARAMS["MIXUP_ENABLED"]:
                lam = np.random.beta(0.2, 0.2)
                rand_index = torch.randperm(batch.size()[0]).to(device)
                batch = lam * batch + (1 - lam) * batch[rand_index]
            
            # --- DENOISING TRICK ---
            noise = torch.randn_like(batch) * PARAMS["DENOISING_FACTOR"]
            noisy_batch = batch + noise
            
            optimizer.zero_grad()
            out = model(noisy_batch) # Feed noisy
            loss = criterion(out, batch) # Grade against clean
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch, _ in val_loader:
                batch = batch.to(device)
                val_loss += criterion(model(batch), batch).item()
        val_loss /= len(val_loader)
        
        history_train.append(train_loss)
        history_val.append(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(MODELS_DIR, f"best_{machine}.pth"))
            status = "Best"
        else:
            patience_counter += 1
            status = f"Wait {patience_counter}/{PARAMS['PATIENCE']}"
            
        print(f"  Ep {epoch:03d} | Tr L1: {train_loss:.5f} | Val L1: {val_loss:.5f} | {status}")
        
        if patience_counter >= PARAMS["PATIENCE"]:
            print("  -> Early Stopping")
            break
            
    # Save Loss Curve
    plt.figure(figsize=(8, 4))
    plt.plot(history_train, label='Train L1')
    plt.plot(history_val, label='Validation L1')
    plt.title(f"{machine} Training Curve")
    plt.legend()
    plt.savefig(os.path.join(RUN_DIR, f"{machine}_loss_curve.png"))
    plt.close()

Device: cuda | Saving to: experiments\runs\run_20260423_091243

TRAINING: ToyCar
  Ep 001 | Tr L1: 0.11219 | Val L1: 0.09448 | Best
  Ep 002 | Tr L1: 0.08583 | Val L1: 0.09013 | Best
  Ep 003 | Tr L1: 0.07916 | Val L1: 0.08370 | Best
  Ep 004 | Tr L1: 0.07816 | Val L1: 0.08183 | Best
  Ep 005 | Tr L1: 0.07542 | Val L1: 0.07988 | Best
  Ep 006 | Tr L1: 0.07401 | Val L1: 0.07793 | Best
  Ep 007 | Tr L1: 0.07218 | Val L1: 0.07709 | Best
  Ep 008 | Tr L1: 0.07279 | Val L1: 0.07728 | Wait 1/20
  Ep 009 | Tr L1: 0.07014 | Val L1: 0.07509 | Best
  Ep 010 | Tr L1: 0.06928 | Val L1: 0.07441 | Best
  Ep 011 | Tr L1: 0.06922 | Val L1: 0.07230 | Best
  Ep 012 | Tr L1: 0.06913 | Val L1: 0.07173 | Best
  Ep 013 | Tr L1: 0.06856 | Val L1: 0.07050 | Best
  Ep 014 | Tr L1: 0.06652 | Val L1: 0.07084 | Wait 1/20
  Ep 015 | Tr L1: 0.06577 | Val L1: 0.06934 | Best
  Ep 016 | Tr L1: 0.06509 | Val L1: 0.06836 | Best
  Ep 017 | Tr L1: 0.06542 | Val L1: 0.06875 | Wait 1/20
  Ep 018 | Tr L1: 0.06225 | Val L1: 0